# 2_models/03 — Mortality risk trajectories (landmark risk scores)

Scores every patient's mortality risk at landmarks months 0, 3, 6 … 60, re-pooling the notes
available up to each landmark and re-scoring **one fixed model**, so each patient's series is a risk
*trajectory* rather than a single baseline score.

**Runs after** `1_data/01` (reads note embeddings and the survival cohort directly, not a prebuilt prediction
dataset) and **before** `4_figures/02` (`figures.prep.figure4` clusters these trajectories into risk slopes).

**Landmarks are independent.** At month *M* the at-risk set is patients whose original, unshifted
`tt_death` exceeds day *M*×30 — drawn from the full baseline cohort, *not* chained through the
previous landmark's survivors, which would compound into a survivor-enriched cohort and make AUCs
incomparable across the figure's x-axis.

**One model, fit once at month 0.** Hyperparameters are selected once on the baseline cohort using
the same grid as the full-cohort runs; every later landmark is inference only. That is what makes a
trajectory interpretable — a rising trajectory means the patient's features moved toward higher
predicted risk, not that the model drifted. The month-0 column is itself a nested-CV held-out score.

### Cost and failure behavior

Expensive: 21 landmarks each re-pool the full embedding array, though model fitting is now one grid
search plus one nested CV rather than 21 of them. **An interrupted run resumes** — each landmark is
checkpointed as it completes, keyed to a fingerprint of the cohort, features, landmarks, seed,
hyperparameter grid and a content hash of the model inputs. A resume does repeat the month-0 grid
search (the selected `(l1_ratio, alpha)` is needed to score anything); the 20 landmark re-scorings
are what it actually saves.

A failed landmark is caught, recorded and skipped, leaving an all-`NaN` column. Both CSVs are still
written, then the script **raises at the end** naming the failed months — so a non-zero exit means
the output has holes, not that there is no output. Set `RETRY_FAILED = True` to refit them.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<22} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def report_outputs(outputs: list[tuple[str, str]]) -> None:
    """Print size and mtime for each (label, path) that exists."""
    for label, path in outputs:
        if os.path.exists(path):
            mb = os.path.getsize(path) / 1e6
            mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(path)))
            print(f"[ok     ] {label:<26} {mb:>8.1f} MB   {mtime}")
        else:
            print(f"[missing] {label:<26} {path}")


def run_module(module: str, args: list[str] | None = None, env: dict | None = None,
               capture: bool = False) -> dict:
    """Run `python -m module` from REPO_ROOT. Returns {returncode, wall_s, stdout}."""
    cmd = [sys.executable, "-m", module, *(args or [])]
    started = time.perf_counter()
    run_env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}
    kwargs = dict(cwd=str(REPO_ROOT), env=run_env)
    if capture:
        kwargs.update(text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    proc = subprocess.run(cmd, **kwargs)
    return {"returncode": proc.returncode, "wall_s": time.perf_counter() - started,
            "stdout": (proc.stdout or "") if capture else ""}


print(f"repo root: {REPO_ROOT}")
print(f"Python:  {sys.executable}")

## Configuration

`DECAY_PARAM` and `MONTHS_TO_TEST` mirror module constants in the script — change them there, not
here. `SLOPE_LANDMARK_MONTHS` / `MIN_SLOPE_POINTS` mirror `figures.prep.figure4` for the hand-off
check.

**Text cohort definition.** The script pools notes with `continuous_window=False`, matching the
canonical text cohort built by `pipelines/preprocessing/generate_embedding_prediction_datasets.py`
— every pre-landmark note of the three types (`Clinician`, `Imaging`, `Pathology`) is pooled. The
function's default (`True`) instead drops notes across gaps > 2 years, which would pool a different
note set and make the month-0 column incomparable to the full-cohort risk scores it reproduces.

`DECAY_PARAM = 0.1` **is** a deliberate departure from the canonical `0.01` — a 10× steeper time
decay, chosen so a landmark's score is dominated by recent notes rather than the patient's whole
history. It is part of the output filenames, so the two parameterizations never overwrite one
another.

In [ ]:
import schemes

MODULE = "pipelines.trajectories.generate_mortality_trajectories"

DECAY_PARAM = 0.1
MONTHS_TO_TEST = [i * 3 for i in range(1, 21)]   # 3..60; month 0 is the baseline column

TRAJECTORY_PATH = os.path.join(schemes.scheme_results_dir("death_met"), "mortality_trajectories")
TRAJECTORY_CSV = os.path.join(TRAJECTORY_PATH, f"survival_trajectories_w_decay_param_{DECAY_PARAM}.csv")
RISK_SETS_CSV = os.path.join(TRAJECTORY_PATH, f"landmark_risk_sets_w_decay_param_{DECAY_PARAM}.csv")
ATTRITION_CSV = os.path.join(TRAJECTORY_PATH, f"cohort_attrition_w_decay_param_{DECAY_PARAM}.csv")
CHECKPOINT_DIR = os.path.join(TRAJECTORY_PATH, "checkpoints")

# figures.prep.figure4 constants, mirrored for the hand-off check.
SLOPE_LANDMARK_MONTHS = 12
MIN_SLOPE_POINTS = 3

ENABLED = True
SKIP_IF_DONE = True    # the run is hours long; turn off to recompute
RESUME = True          # False -> ignore checkpoints and refit every landmark
RETRY_FAILED = False   # True  -> also refit landmarks a previous run recorded as failed

print(f"output dir: {TRAJECTORY_PATH}")
print(f"landmarks:  0, {', '.join(str(m) for m in MONTHS_TO_TEST[:3])} ... "
      f"{MONTHS_TO_TEST[-1]} months  ({len(MONTHS_TO_TEST) + 1} columns)")

## Preconditions

The embedding array is the large one. This cell does not raise.

In [ ]:
check_inputs([
    ("note embeddings meta",  os.path.join(config.NOTES_PATH,
                                           "full_clinical_notes_embeddings_metadata.parquet")),
    ("note embeddings array", os.path.join(config.NOTES_PATH,
                                           "full_clinical_notes_embeddings_as_array.npy.zst")),
    ("survival cohort",       os.path.join(config.SURV_PATH, "death_met_surv_df.parquet")),
    ("cancer types",          os.path.join(config.FEATURE_PATH, "cancer_type_df.csv.gz")),
])

try:
    import zstandard  # noqa: F401
    print("[ok ] zstandard importable (needed to decompress the embedding array)")
except ImportError:
    print("[MISSING] zstandard not importable — run this on the cluster kernel.")

## Run

Output streams straight through — this runs for hours, so silence would be indistinguishable from a
hang. The script reports at landmark, outer-fold and inner-grid level:

```
[10:32:19] landmark 6/20 (month 18): 48 patients at risk
[10:32:19]   pooling embeddings (window=540d) ...
[outer] 1/5 folds complete (l1_ratio=1, alpha=0.001)
[CV] [######------------------------] 50/250 fold×hyperparameter fits (20%)
[10:41:02]   month 18 done in 8.7 min (6/20 landmarks, 1.32 h elapsed)
```

The per-landmark completion line carries a running elapsed total, so you can extrapolate the finish
time after two or three landmarks. Set `TRAJECTORY_PROGRESS=0` to silence it.

In [ ]:
if not ENABLED:
    print("=== disabled ===")
elif SKIP_IF_DONE and os.path.exists(TRAJECTORY_CSV):
    mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(TRAJECTORY_CSV)))
    print(f"=== already done (trajectory CSV written {mtime}) ===\n    {TRAJECTORY_CSV}")
    print("    Set SKIP_IF_DONE = False to recompute.")
else:
    if RESUME and os.path.isdir(CHECKPOINT_DIR):
        print("[resume] checkpoint store present — completed landmarks will be reloaded")
    elif not RESUME:
        print("[resume] disabled — every landmark will be refit")

    print(f"{'=' * 78}\n=== python -m {MODULE}\n{'=' * 78}", flush=True)
    outcome = run_module(MODULE, env={
        "TRAJECTORY_RESUME": "1" if RESUME else "0",
        "TRAJECTORY_RETRY_FAILED": "1" if RETRY_FAILED else "0",
    })
    hours = outcome["wall_s"] / 3600

    if outcome["returncode"] == 0:
        print(f"\n[ok] all {len(MONTHS_TO_TEST) + 1} landmarks scored in {hours:.2f} h")
    elif os.path.exists(TRAJECTORY_CSV):
        print(f"\n[exit {outcome['returncode']}] finished in {hours:.2f} h with at least one failed "
              "landmark. Both CSVs were written before the raise, so the output exists with holes — "
              "the coverage table below says which. Set RETRY_FAILED = True to refit them.")
    else:
        print(f"\n[exit {outcome['returncode']}] failed in {hours:.2f} h without writing anything — "
              "the run died before the landmark loop finished (see the traceback above). Any "
              "landmarks that did complete are checkpointed.")

## Landmark coverage

How many patients got a score at each landmark, against the at-risk denominator the script recorded.
`n_scored` well below `n_at_risk` means patients were at risk but dropped for missing embeddings or
covariates; `n_scored = 0` is a failed or skipped landmark. Guarded — safe on a partial pipeline.

**The denominator here is already post-attrition.** `n_at_risk` counts patients in the *baseline
cohort*, which the script forms by pooling embeddings, inner-joining `cancer_type_df`, and dropping
incomplete cases. Patients lost at those steps never reach this table, so 100% coverage here means
full coverage of whatever cohort survived them — not of the survival cohort on disk. The run log
prints the attrition ladder (`survival cohort` → `after embedding pooling` → `after cancer-type
join` → `after complete-case filter` → `baseline cohort`); read it there, and note the `[warn]` line
if the cancer-type join dropped anyone. The cell below re-reports that ladder from the checkpoint
metadata when the run wrote it.

In [ ]:
import polars as pl

report_outputs([("trajectory scores", TRAJECTORY_CSV), ("landmark risk sets", RISK_SETS_CSV)])

manifest = os.path.join(CHECKPOINT_DIR, "landmark_manifest.csv")
if os.path.exists(manifest):
    rows = pl.read_csv(manifest)
    done = rows.filter(pl.col("status") == "done")
    failed_ckpt = rows.filter(pl.col("status") == "failed")
    print(f"[ok     ] {'checkpoints':<26} {done.height} reusable, {failed_ckpt.height} failed")
    for month, reason in failed_ckpt.select("month", "reason").iter_rows():
        print(f"           failed month {month}: {reason}")
    if failed_ckpt.height:
        print("           set RETRY_FAILED = True to refit these")
else:
    print(f"[missing] {'checkpoints':<26} {CHECKPOINT_DIR}")

# Cohort attrition upstream of the landmark denominators. Every drop here happens before
# `cohort_mrns` is taken, so it is invisible to the per-landmark table below: without this,
# a large cancer-type-join loss would still read as 100% coverage.
if os.path.exists(ATTRITION_CSV):
    att = pl.read_csv(ATTRITION_CSV)
    print("\nCohort attrition (upstream of every n_at_risk below)\n")
    prev = None
    for step, n in att.iter_rows():
        drop = "" if prev is None else f"  -{prev - n:,}"
        print(f"  {step:<28}{n:>8,}{drop}")
        prev = n
    first, last = att["n_patients"][0], att["n_patients"][-1]
    if first:
        print(f"\n  retained {last:,}/{first:,} ({100.0 * last / first:.1f}%) of the survival cohort")
    joined = dict(zip(att["step"].to_list(), att["n_patients"].to_list()))
    n_pool, n_type = joined.get("after embedding pooling"), joined.get("after cancer-type join")
    if n_pool and n_type is not None and n_type < n_pool:
        print(f"  [warn] cancer-type join dropped {n_pool - n_type:,} "
              f"({100.0 * (n_pool - n_type) / n_pool:.1f}%) — these never appear below")
else:
    print(f"\n[missing] {'cohort attrition':<26} {ATTRITION_CSV}")
    print("           (written by runs after the continuous_window alignment; "
          "re-run to populate)")

def _month_num(col: str) -> int:
    return int(col.split("_")[1])


month_cols, scored, traj, n_rows = [], {}, None, 0
if not os.path.exists(TRAJECTORY_CSV):
    print("\nTrajectory CSV not written — run has not completed.")
else:
    traj = pl.read_csv(TRAJECTORY_CSV)
    n_rows = traj.height
    month_cols = sorted((c for c in traj.columns if c != "DFCI_MRN"), key=_month_num)
    scored = {
        c: int(traj.select(pl.col(c).cast(pl.Float64, strict=False).is_finite().sum()).item())
        for c in month_cols
    }

    at_risk = {}
    if os.path.exists(RISK_SETS_CSV):
        rs = pl.read_csv(RISK_SETS_CSV)
        at_risk = dict(zip(rs["months"].to_list(), rs["n_at_risk"].to_list()))
    else:
        print("\n[note] landmark_risk_sets CSV missing — at-risk denominators unavailable.")

    print(f"\nCohort: {n_rows:,} patients, {len(month_cols)} landmark columns\n")
    print(f"  {'month':>5}  {'n_scored':>9}  {'n_at_risk':>9}  {'% cohort':>9}   status")
    empty_months = []
    for c in month_cols:
        m, n = _month_num(c), scored[c]
        ar = at_risk.get(m)
        pct = 100.0 * n / n_rows if n_rows else 0.0
        if n == 0:
            status = "EMPTY (failed or skipped)"
            empty_months.append(m)
        elif pct <= 10.0:
            status = "below figure4's >10% keep threshold"
        else:
            status = ""
        print(f"  {m:>5}  {n:>9,}  {(f'{ar:,}' if ar is not None else '-'):>9}  {pct:>8.1f}%   {status}")

    if empty_months:
        print(f"\n{len(empty_months)} empty landmark(s): "
              f"{', '.join(str(m) for m in empty_months)} months.")

## Hand-off to 4_figures/02

`figures.prep.figure4` keeps landmark columns observed in >10% of rows, restricts to months
0..`SLOPE_LANDMARK_MONTHS`, and requires `MIN_SLOPE_POINTS` observed values per patient in that
window. This applies the same three rules so you know what Figure 4 will have to work with.

In [ ]:
n_slope = 0
if traj is None:
    print("Trajectory CSV not written — not ready for 4_figures/02.")
else:
    kept = [c for c in month_cols if scored[c] > 0.1 * n_rows]
    in_window = [c for c in kept if _month_num(c) <= SLOPE_LANDMARK_MONTHS]

    print(f"landmark columns written:            {len(month_cols)}")
    print(f"  kept by figure4 (>10% observed):   {len(kept)}")
    print(f"  within slope window (0..{SLOPE_LANDMARK_MONTHS} months): {len(in_window)}"
          f"   [{', '.join(str(_month_num(c)) for c in in_window)}]")

    if len(in_window) < MIN_SLOPE_POINTS:
        print(f"\n[FAIL] Only {len(in_window)} usable landmark(s) in the slope window, but "
              f"MIN_SLOPE_POINTS = {MIN_SLOPE_POINTS}. No patient can get a risk slope.")
    else:
        observed = traj.select([
            pl.col(c).cast(pl.Float64, strict=False).is_finite().cast(pl.Int32) for c in in_window
        ]).sum_horizontal()
        n_slope = int((observed >= MIN_SLOPE_POINTS).sum())
        print(f"\npatients with >= {MIN_SLOPE_POINTS} observed landmarks in the window: "
              f"{n_slope:,} / {n_rows:,}  ({100.0 * n_slope / n_rows:.1f}%)")

    print(f"\nfigure data dir: {config.FIGURE_DATA_DIR}")
    report_outputs([(csv, os.path.join(config.FIGURE_DATA_DIR, csv)) for csv in (
        "fig4_trajectories_heatmap.csv", "fig4_km_data.csv", "fig4_cluster_severity.csv",
        "fig4_group_trajectories.csv", "fig4_slope_by_stage.csv", "fig4_silhouette.csv")])

    print("\n" + ("Ready for 4_figures/02." if n_slope else
                  "Not ready — re-run the failed landmarks before 4_figures/02."))